In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 # İstediğin değeri seçebilirsin, Unsloth RoPE ölçeklendirmesini otomatik destekler!
dtype = None # Otomatik tespit için None.
load_in_4bit = True # Bellek kullanımını azaltmak için 4-bit quantizasyon.

# Modeli Llama-3.2-3B'nin 4-bit sürümü ile değiştiriyoruz
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.5.4: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.5.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
## Data Prep
# Veri seti ve prompt şablonu

In [ ]:
# Llama-3 Chat Şablonu
llama3_chat_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{}<|eot_id|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{}<|eot_id|>"""

# Modelin aklını karıştırmayacak net ve kısa sistem komutumuz
sabit_system_prompt = "Sen, kripto para ve borsa konularında uzmanlaşmış, Türkiye ve global piyasalara hakim bir finans asistanısın. Sorulara doğrudan, net ve eğitici cevaplar verirsin."

def formatting_prompts_func(examples):
    users      = examples["user"]
    assistants = examples["assistant"]
    texts = []

    # Veri setindeki bozuk 'system' kolonunu görmezden gelip kendi promptumuzu basıyoruz
    for user, assistant in zip(users, assistants):
        text = llama3_chat_prompt.format(sabit_system_prompt, user, assistant)
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset

# Veri setini yükle ve formatla
dataset = load_dataset("AlicanKiraz0/Turkish-Finance-SFT-Dataset", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
## Model Eğitimi (Model Train)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import torch

# GPU belleğini temizleyelim (Önceki çöken çalışmadan kalan çöpleri temizler)
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1024, # 2048 ise 1024'e düşürmeyi dene
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # 2'den 1'e düşürüldü
        gradient_accumulation_steps = 8, # 4'ten 8'e çıkarıldı (1x8 = 8 efektif batch)
        warmup_steps = 5,
        #num_train_epochs = 1,
        max_steps=150,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/2397 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,397 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,2.740962
2,2.652345
3,2.673409
4,2.589623
5,2.577005
6,2.512549
7,2.403306
8,2.353508
9,2.379173
10,2.362794


Step,Training Loss
1,2.740962
2,2.652345
3,2.673409
4,2.589623
5,2.577005
6,2.512549
7,2.403306
8,2.353508
9,2.379173
10,2.362794


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-150/tokenizer_config.json.


In [ ]:
# Modeli ve tokenizer'ı yerel diske (Colab dosyalarına) kaydeder
model.save_pretrained("finans_asistani_lora")
tokenizer.save_pretrained("finans_asistani_lora")

print("Model başarıyla kaydedildi!")

Unsloth: Restored added_tokens_decoder metadata in finans_asistani_lora/tokenizer_config.json.


Model başarıyla kaydedildi!


In [ ]:
# 1. Modeli hızlı çıkarım moduna alıyoruz
FastLanguageModel.for_inference(model)

# 2. Yeni, kısa ve net sistem komutumuz
sabit_system_prompt = "Sen, kripto para ve borsa konularında uzmanlaşmış, Türkiye ve global piyasalara hakim bir finans asistanısın. Sorulara doğrudan, net ve eğitici cevaplar verirsin."

# 3. Modele sormak istediğin soru
soru = "Stop-loss stratejisi nasıl uygulanır?"

# 4. Promptu hazırlıyoruz
inputs = tokenizer(
[
    llama3_chat_prompt.format(
        sabit_system_prompt,
        soru,
        "", # Asistanın cevabı üretmesi için boş bırakıyoruz
    )
], return_tensors = "pt").to("cuda")

# 5. Cevabı kelime kelime ekrana yazdırmak için Streamer kullanıyoruz
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print(f"SORU: {soru}\n")
print("ASİSTANIN CEVABI:")
print("-" * 50)

# 6. Üretimi başlatıyoruz
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.3,
    repetition_penalty = 1.05
)

SORU: Stop-loss stratejisi nasıl uygulanır?

ASİSTANIN CEVABI:
--------------------------------------------------
<think>
Stop-loss, kripto para ve borsa yatırımcıları için önemli bir risk management aracıdır. Stop-loss, belirli bir fiyat seviyesinde (stop-loss seviyesi) işlem yapılırsa, pozisyonu kapatmak için kullanılan bir mekanizmadır. Bu, fiyatın belirli bir sınıra yaklaştığında (örneğin %5-10% düşüş) stop-loss seviyesini geçmesi durumunda, pozisyonun otomatik olarak kapatılması anlamına gelir.

Stop-loss stratejisini uygularken, aşağıdaki adımları takip edersin:

1. **Stop-loss seviyesi belirle**: İlk olarak, bir işlemde stop-loss seviyesi belirlemek için hangi kriterleri kullanacaksın? Genellikle, işlem fiyatının belirli bir % düşüşüne (örneğin %5-10) ulaşması durumunda stop-loss seviyesi belirlenir. Örneğin, bir işlemde stop-loss %5 seviyesi ise, fiyat %5 düşerse stop-loss otomatik olarak kapatılır.

2. **Stop-loss seviyesini fiyatın altındaki seviyeye koy**: Stop-loss seviyesi

In [ ]:
# 1. Modeli hızlı çıkarım moduna alıyoruz
FastLanguageModel.for_inference(model)

# 2. Yeni, kısa ve net sistem komutumuz
sabit_system_prompt = "Sen, kripto para ve borsa konularında uzmanlaşmış, Türkiye ve global piyasalara hakim bir finans asistanısın. Sorulara doğrudan, net ve eğitici cevaplar verirsin."

# 3. Modele sormak istediğin soru
soru = "merhaba"

# 4. Promptu hazırlıyoruz
inputs = tokenizer(
[
    llama3_chat_prompt.format(
        sabit_system_prompt,
        soru,
        "", # Asistanın cevabı üretmesi için boş bırakıyoruz
    )
], return_tensors = "pt").to("cuda")

# 5. Cevabı kelime kelime ekrana yazdırmak için Streamer kullanıyoruz
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print(f"SORU: {soru}\n")
print("ASİSTANIN CEVABI:")
print("-" * 50)

# 6. Üretimi başlatıyoruz
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.3,
    repetition_penalty = 1.05
)

SORU: merhaba

ASİSTANIN CEVABI:
--------------------------------------------------
Merhaba! Sorununuzu yazmadım. Lütfen soruyu yazıp, hangi konuyu (kripto, borsa, hisse senedi, faiz, nakit akışı vb.) ve hangi soruyu soruyorsun? Cevap verirken net ve eğitici olacağınıaim.<|eot_id|>


In [ ]:
# 1. Modeli ve tokenizer'ı Colab içine klasör olarak kaydediyoruz
model.save_pretrained("finans_asistani_v2_model")
tokenizer.save_pretrained("finans_asistani_v2_model")

from huggingface_hub import login, HfApi

# 2. Classic Write yetkili Token'ın ile giriş yap
hf_token = "hf_TOKENINI_BURAYA_YAZ" # Kendi token'ını eklemeyi unutma
login(token=hf_token)

# 3. Yükleme işlemini başlatıyoruz
api = HfApi()
print("Yükleme başlıyor, lütfen bekleyin...")

api.upload_folder(
    folder_path="finans_asistani_v2_model",
    repo_id="IsmailAI/finans_asistani_3b_v2", # Eski bozuk modelle karışmaması için v2 dedik
    repo_type="model"
)
print("✅ Başarıyla yüklendi!")

In [ ]:
## modeli gguf formatında Hugginface göndermeye çalışırken bellekle ilgili hata alınıyorsa model.push_to_gguf yerine şu yolu izle
# 3. Alternatif Yol: 16-Bit Yükleyip GGUF'a Çevirme
# Eğer Colab ortamı GGUF çevirisinde inatla hata vermeye devam ediyorsa, modeli standart (saf) formatta Hugging Face'e yükleyip, GGUF dönüşümünü Hugging Face'in kendi sunucularına yaptırabilirsin.
#Önce modeli 16-bit formatında yükle:

In [ ]:
hf_token = "hf_tokeninizi girin" # Write yetkili token'ını buraya gir

# Modeli GGUF yerine doğrudan birleştirilmiş (merged) 16-bit model olarak yükler
model.push_to_hub_merged(
    "IsmailAI/finans_asistani_3b_v2",
    tokenizer,
    save_method = "merged_16bit",
    token = hf_token,
)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in IsmailAI/finans_asistani_3b_v2/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tani_3b_v2/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          | 4.26MB / 1.46GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [05:35<00:00, 167.56s/it]


Unsloth: Merge process complete. Saved to `/content/IsmailAI/finans_asistani_3b_v2`


* Model başarıyla Hugging Face profiline yüklendikten sonra:

Hugging Face'te arama çubuğuna GGUF My Repo yaz. (Veya şu linke git: https://huggingface.co/spaces/ggml-org/gguf-my-repo)

Bu araç senden sadece Hugging Face repo ismini (örn: IsmailAI/finans_asistani_3b) isteyecek.

Araca repo ismini verdiğinde, Hugging Face arka planda modeli senin için ücretsiz olarak GGUF'a çevirip profiline otomatik olarak yükleyecektir.